# COMP6252 Coursework 1

This notebook is used as the main workflow for the coursework experiments. 
To keep the notebook readable, the model definitions, dataset classes, training loop, evaluation code, GAN augmentation, and plotting functions are implemented in `music_genre_source.py`.

The six required neural architectures are imported and used in order:

- `Net1`: fully connected network with two hidden layers
- `Net2`: convolutional neural network
- `Net3`: CNN with batch normalisation
- `Net4`: same as Net3, trained with RMSProp
- `Net5`: LSTM network for audio feature sequences
- `Net6`: same LSTM architecture as Net5 with GAN-based data augmentation

The notebook manages the experiment sequence, including data loading, 70/20/10 splitting, training, evaluation, and plotting accuracy/error curves, and confusion matrix which can be seen at `\outputs`.

In [ ]:
from pathlib import Path
import torch

from music_genre_source import (
    AudioFeatureSequenceDataset,
    IMAGE_EPOCH_OPTIONS,
    IMAGE_SIZE,
    Net1,
    Net2,
    Net3,
    Net4,
    Net5,
    Net6,
    ResizeToTensor,
    augment_with_gan,
    resolve_dataset_roots,
    save_result,
    seed_everything,
    split_dataset,
    train_classifier,
    ImageFolderDataset,
)

from torch.utils.data import DataLoader

In [2]:
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset_root = Path('dataset/gtzan-dataset-music-genre-classification')
output_dir = Path('outputs')
image_root, audio_root = resolve_dataset_roots(dataset_root)
print(device)
print(image_root)
print(audio_root)



cuda
dataset\gtzan-dataset-music-genre-classification\images_original
dataset\gtzan-dataset-music-genre-classification\genres_original


## Image Models (Net1-Net4)

In [3]:
transform = ResizeToTensor(IMAGE_SIZE)
image_dataset = ImageFolderDataset(image_root, transform=transform)
train_set, val_set, test_set = split_dataset(image_dataset, seed=42)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

In [4]:
image_models = [
    ('Net1', Net1, 'adam'),
    ('Net2', Net2, 'adam'),
    ('Net3', Net3, 'adam'),
    ('Net4', Net4, 'rmsprop'),
]

for model_name, model_cls, optimizer_name in image_models:
    for epochs in IMAGE_EPOCH_OPTIONS:
        model = model_cls(num_classes=len(image_dataset.classes))
        result, extras = train_classifier(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            device=device,
            epochs=epochs,
            optimizer_name=optimizer_name,
            learning_rate=1e-4,
            class_names=image_dataset.classes,
        )
        result.model_name = model_name
        save_result(output_dir, result, extras)

Net1:   0%|          | 0/50 [00:00<?, ?it/s]

Net1 epoch 001/50 train_loss=2.8655 val_loss=2.3222 train_acc=0.106 val_acc=0.151 best_val_acc=0.151
Net1 epoch 010/50 train_loss=2.1471 val_loss=2.1024 train_acc=0.180 val_acc=0.231 best_val_acc=0.231
Net1 epoch 020/50 train_loss=2.0715 val_loss=2.0302 train_acc=0.193 val_acc=0.221 best_val_acc=0.246
Net1 epoch 030/50 train_loss=2.0205 val_loss=1.9882 train_acc=0.212 val_acc=0.231 best_val_acc=0.246
Net1 epoch 040/50 train_loss=1.9842 val_loss=1.9049 train_acc=0.246 val_acc=0.296 best_val_acc=0.312
Net1 epoch 050/50 train_loss=1.8993 val_loss=1.8879 train_acc=0.269 val_acc=0.307 best_val_acc=0.352


Net1:   0%|          | 0/100 [00:00<?, ?it/s]

Net1 epoch 001/100 train_loss=2.8078 val_loss=2.2408 train_acc=0.129 val_acc=0.186 best_val_acc=0.186
Net1 epoch 010/100 train_loss=2.1553 val_loss=2.0619 train_acc=0.162 val_acc=0.216 best_val_acc=0.236
Net1 epoch 020/100 train_loss=2.0849 val_loss=2.0218 train_acc=0.199 val_acc=0.251 best_val_acc=0.261
Net1 epoch 030/100 train_loss=2.0229 val_loss=1.9901 train_acc=0.202 val_acc=0.231 best_val_acc=0.266
Net1 epoch 040/100 train_loss=1.9823 val_loss=1.9517 train_acc=0.209 val_acc=0.286 best_val_acc=0.286
Net1 epoch 050/100 train_loss=1.8994 val_loss=1.9216 train_acc=0.258 val_acc=0.261 best_val_acc=0.327
Net1 epoch 060/100 train_loss=1.8339 val_loss=1.9051 train_acc=0.273 val_acc=0.276 best_val_acc=0.357
Net1 epoch 070/100 train_loss=1.7895 val_loss=1.8118 train_acc=0.292 val_acc=0.352 best_val_acc=0.387
Net1 epoch 080/100 train_loss=1.7178 val_loss=1.7652 train_acc=0.320 val_acc=0.357 best_val_acc=0.387
Net1 epoch 090/100 train_loss=1.6854 val_loss=1.7434 train_acc=0.346 val_acc=0.382

Net2:   0%|          | 0/50 [00:00<?, ?it/s]

...

Net3 epoch 001/100 train_loss=2.2201 val_loss=2.0979 train_acc=0.179 val_acc=0.256 best_val_acc=0.256
Net3 epoch 010/100 train_loss=1.3125 val_loss=1.3929 train_acc=0.548 val_acc=0.548 best_val_acc=0.553
Net3 epoch 020/100 train_loss=1.0562 val_loss=1.1171 train_acc=0.654 val_acc=0.643 best_val_acc=0.643
Net3 epoch 030/100 train_loss=0.8999 val_loss=1.2054 train_acc=0.687 val_acc=0.613 best_val_acc=0.648
Net3 epoch 040/100 train_loss=0.7862 val_loss=1.2294 train_acc=0.725 val_acc=0.608 best_val_acc=0.663
Net3 epoch 050/100 train_loss=0.6864 val_loss=1.0369 train_acc=0.758 val_acc=0.638 best_val_acc=0.678
Net3 epoch 060/100 train_loss=0.5861 val_loss=0.9501 train_acc=0.820 val_acc=0.688 best_val_acc=0.698
Net3 epoch 070/100 train_loss=0.5070 val_loss=0.9773 train_acc=0.838 val_acc=0.683 best_val_acc=0.714
Net3 epoch 080/100 train_loss=0.4479 val_loss=0.9766 train_acc=0.863 val_acc=0.704 best_val_acc=0.714
Net3 epoch 090/100 train_loss=0.3406 val_loss=0.9628 train_acc=0.894 val_acc=0.683

Net4:   0%|          | 0/50 [00:00<?, ?it/s]

Net4 epoch 001/50 train_loss=2.2921 val_loss=2.0174 train_acc=0.162 val_acc=0.221 best_val_acc=0.221
Net4 epoch 010/50 train_loss=1.3592 val_loss=2.1856 train_acc=0.508 val_acc=0.276 best_val_acc=0.477
Net4 epoch 020/50 train_loss=1.0690 val_loss=1.9900 train_acc=0.618 val_acc=0.372 best_val_acc=0.573
Net4 epoch 030/50 train_loss=0.9001 val_loss=1.2190 train_acc=0.658 val_acc=0.608 best_val_acc=0.613
Net4 epoch 040/50 train_loss=0.7394 val_loss=1.3141 train_acc=0.735 val_acc=0.623 best_val_acc=0.623
Net4 epoch 050/50 train_loss=0.5694 val_loss=1.4406 train_acc=0.794 val_acc=0.643 best_val_acc=0.688


Net4:   0%|          | 0/100 [00:00<?, ?it/s]

Net4 epoch 001/100 train_loss=2.3607 val_loss=2.0581 train_acc=0.163 val_acc=0.226 best_val_acc=0.226
Net4 epoch 010/100 train_loss=1.3257 val_loss=1.5885 train_acc=0.531 val_acc=0.427 best_val_acc=0.513
Net4 epoch 020/100 train_loss=1.0011 val_loss=1.2877 train_acc=0.657 val_acc=0.568 best_val_acc=0.593
Net4 epoch 030/100 train_loss=0.8926 val_loss=1.5652 train_acc=0.681 val_acc=0.508 best_val_acc=0.623
Net4 epoch 040/100 train_loss=0.6653 val_loss=1.2320 train_acc=0.755 val_acc=0.608 best_val_acc=0.658
Net4 epoch 050/100 train_loss=0.6270 val_loss=1.3257 train_acc=0.773 val_acc=0.633 best_val_acc=0.658
Net4 epoch 060/100 train_loss=0.4497 val_loss=1.5246 train_acc=0.850 val_acc=0.628 best_val_acc=0.658
Net4 epoch 070/100 train_loss=0.3332 val_loss=1.4044 train_acc=0.890 val_acc=0.668 best_val_acc=0.688
Net4 epoch 080/100 train_loss=0.2992 val_loss=1.2883 train_acc=0.898 val_acc=0.688 best_val_acc=0.688
Net4 epoch 090/100 train_loss=0.2377 val_loss=2.2649 train_acc=0.908 val_acc=0.568

## Audio Models (Net5-Net6)

In [5]:
audio_dataset = AudioFeatureSequenceDataset.from_dataset_root(dataset_root)
train_audio, val_audio, test_audio = split_dataset(audio_dataset, seed=42)
audio_dataset.fit_standardizer(train_audio.indices)

train_audio_loader = DataLoader(train_audio, batch_size=16, shuffle=True)
val_audio_loader = DataLoader(val_audio, batch_size=16, shuffle=False)
test_audio_loader = DataLoader(test_audio, batch_size=16, shuffle=False)

In [6]:
net5 = Net5(input_size=audio_dataset.n_features, hidden_size=64, num_classes=len(audio_dataset.classes))
result5, extras5 = train_classifier(
    model=net5,
    train_loader=train_audio_loader,
    val_loader=val_audio_loader,
    test_loader=test_audio_loader,
    device=device,
    epochs=80,
    optimizer_name='adam',
    learning_rate=1e-4,
    class_names=audio_dataset.classes,
)
result5.model_name = 'Net5'
save_result(output_dir, result5, extras5)

Net5:   0%|          | 0/80 [00:00<?, ?it/s]

Net5 epoch 001/80 train_loss=2.2940 val_loss=2.2780 train_acc=0.100 val_acc=0.155 best_val_acc=0.155
Net5 epoch 010/80 train_loss=1.4892 val_loss=1.4658 train_acc=0.473 val_acc=0.460 best_val_acc=0.460
Net5 epoch 020/80 train_loss=0.7327 val_loss=0.9515 train_acc=0.734 val_acc=0.680 best_val_acc=0.690
Net5 epoch 030/80 train_loss=0.3636 val_loss=0.8495 train_acc=0.890 val_acc=0.740 best_val_acc=0.740
Net5 epoch 040/80 train_loss=0.1857 val_loss=0.9356 train_acc=0.951 val_acc=0.720 best_val_acc=0.750
Net5 epoch 050/80 train_loss=0.0859 val_loss=1.0281 train_acc=0.983 val_acc=0.755 best_val_acc=0.755
Net5 epoch 060/80 train_loss=0.0432 val_loss=1.1515 train_acc=0.993 val_acc=0.750 best_val_acc=0.760
Net5 epoch 070/80 train_loss=0.0359 val_loss=1.2950 train_acc=0.991 val_acc=0.725 best_val_acc=0.765
Net5 epoch 080/80 train_loss=0.0206 val_loss=1.2955 train_acc=0.997 val_acc=0.750 best_val_acc=0.765


In [7]:
augmented_train = augment_with_gan(
    train_audio,
    num_classes=len(audio_dataset.classes),
    device=device,
    gan_waveform_length=16384,
    gan_epochs=30,
)
augmented_loader = DataLoader(augmented_train, batch_size=16, shuffle=True)
net6 = Net6(input_size=audio_dataset.n_features, hidden_size=64, num_classes=len(audio_dataset.classes))
result6, extras6 = train_classifier(
    model=net6,
    train_loader=augmented_loader,
    val_loader=val_audio_loader,
    test_loader=test_audio_loader,
    device=device,
    epochs=80,
    optimizer_name='adam',
    learning_rate=1e-4,
    class_names=audio_dataset.classes,
)
result6.model_name = 'Net6'
save_result(output_dir, result6, extras6)

GAN:   0%|          | 0/30 [00:00<?, ?it/s]

GAN epoch 001/30 gen_loss=0.807
GAN epoch 010/30 gen_loss=4.826
GAN epoch 020/30 gen_loss=5.114
GAN epoch 030/30 gen_loss=6.033


Net6:   0%|          | 0/80 [00:00<?, ?it/s]

Net6 epoch 001/80 train_loss=2.2886 val_loss=2.2814 train_acc=0.124 val_acc=0.175 best_val_acc=0.175
Net6 epoch 010/80 train_loss=1.2886 val_loss=1.3726 train_acc=0.506 val_acc=0.525 best_val_acc=0.525
Net6 epoch 020/80 train_loss=0.8107 val_loss=1.0767 train_acc=0.698 val_acc=0.675 best_val_acc=0.675
Net6 epoch 030/80 train_loss=0.5510 val_loss=1.0530 train_acc=0.794 val_acc=0.705 best_val_acc=0.715
Net6 epoch 040/80 train_loss=0.4445 val_loss=1.1130 train_acc=0.824 val_acc=0.705 best_val_acc=0.725
Net6 epoch 050/80 train_loss=0.3325 val_loss=1.1936 train_acc=0.858 val_acc=0.710 best_val_acc=0.725
Net6 epoch 060/80 train_loss=0.2658 val_loss=1.3139 train_acc=0.899 val_acc=0.700 best_val_acc=0.725
Net6 epoch 070/80 train_loss=0.2321 val_loss=1.4461 train_acc=0.916 val_acc=0.685 best_val_acc=0.725
Net6 epoch 080/80 train_loss=0.2052 val_loss=1.4646 train_acc=0.915 val_acc=0.705 best_val_acc=0.725
